In [0]:
raw_users=spark.table("default.bronze_user_table")

In [0]:
raw_users.printSchema() 

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

# Check the current volume has the new dataframe data and , if not exist insert new data raws otherwise put same it is.
def incremental_upsert(dest_table: str, df: DataFrame, unique_key: str, update_at: str, full_refresh=False):
    """
    Performs incremental upsert using updated_at as the cursor value with unique_key, Doesn't support deletes, very minimal.
    """
    if not spark.catalog.tableExists(dest_table) or full_refresh:
        (
            df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(dest_table)
        )
    else:
        last_max = (
            spark.table(dest_table)
            .agg(F.max(update_at).alias("max_ts"))
            .collect()[0]["max_ts"]
        )
        incremental_df = df.filter(F.col(update_at) > last_max)
        if incremental_df.count() > 0:
            delta_table = DeltaTable.forName(spark, dest_table)
            (
                delta_table.alias("t")
                .merge(
                    source=incremental_df.alias("s"),
                    condition=f"s.{unique_key}=t.{unique_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
    
dest_table = "default.stg_users"  # target
incremental_upsert(dest_table, raw_users, "Id", "CreationDate")
